# 07 — Team Aggregation
Aggregates per-player composite scores to one row per match, with home-team
and away-team features side-by-side. Output feeds directly into the
Win/Draw/Loss classifier.

| Output | Path |
|--------|------|
| `match_features_train.parquet` | `data/processed/` |
| `match_features_test.parquet`  | `data/processed/` |

**Target column:** `outcome` — `H` (home win), `D` (draw), `A` (away win).


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DS_DIR   = Path('../data/processed/datasets')
PROC_DIR = Path('../data/processed')

TRAIN_SEASONS = {'2021_2022', '2022_2023', '2023_2024'}
TEST_SEASONS  = {'2024_2025'}

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)
print('✓ Ready')


✓ Ready


## 1 — Load scaled datasets

In [2]:
of_train = pd.read_parquet(DS_DIR / 'outfield_train_scaled.parquet')
of_test  = pd.read_parquet(DS_DIR / 'outfield_test_scaled.parquet')
gk_train = pd.read_parquet(DS_DIR / 'gk_train_scaled.parquet')
gk_test  = pd.read_parquet(DS_DIR / 'gk_test_scaled.parquet')

# Combine for rolling computation (split again at export)
outfield = pd.concat([of_train, of_test], ignore_index=True)
gk       = pd.concat([gk_train, gk_test], ignore_index=True)

print(f'Outfield: {outfield.shape}')
print(f'GK:       {gk.shape}')
print(f'Seasons:  {sorted(outfield["season"].unique())}')


Outfield: (20036, 87)
GK:       (2032, 89)
Seasons:  ['2021_2022', '2022_2023', '2023_2024', '2024_2025']


## 2 — Rolling team strength features
`team_goals_scored` and `opp_goals_scored` in the player data are raw
match-level values (leakage if used directly). We compute a 5-match
rolling average at the team level, sorted by date, using `.shift(1)` so
the current match is excluded from its own window.


In [3]:
# Unique team-match observations (same for all players in team)
team_match = (
    outfield[['match_id', 'team_name', 'match_date', 'season',
              'team_goals_scored', 'opp_goals_scored']]
    .drop_duplicates(subset=['match_id', 'team_name'])
    .sort_values(['team_name', 'match_date'])
    .reset_index(drop=True)
)

# Rolling 5-match avg goals scored / conceded per team
for col, new_col in [('team_goals_scored', 'roll5_attack_str'),
                     ('opp_goals_scored',  'roll5_defence_str')]:
    team_match[new_col] = (
        team_match.groupby('team_name')[col]
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    )

# Fill first-match NaNs with overall mean (computed on all data — benign, no future leakage)
team_match['roll5_attack_str']  = team_match['roll5_attack_str'].fillna(
    team_match['roll5_attack_str'].mean())
team_match['roll5_defence_str'] = team_match['roll5_defence_str'].fillna(
    team_match['roll5_defence_str'].mean())

print('Team-match strength table:', team_match.shape)
print(team_match[['team_name','match_date','team_goals_scored',
                  'roll5_attack_str','roll5_defence_str']].head(8).to_string())


Team-match strength table: (2103, 8)
         team_name                match_date  team_goals_scored  roll5_attack_str  roll5_defence_str
0  AFC Bournemouth 2022-08-27 14:00:00+00:00              3.000             1.171              1.168
1  AFC Bournemouth 2022-08-31 18:30:00+00:00              2.000             3.000              2.000
2  AFC Bournemouth 2022-10-01 14:00:00+00:00              0.000             2.500              1.000
3  AFC Bournemouth 2022-10-08 14:00:00+00:00              2.000             1.667              0.667
4  AFC Bournemouth 2022-10-19 18:30:00+00:00              2.000             1.750              0.500
5  AFC Bournemouth 2022-10-24 19:00:00+00:00              0.000             1.800              0.600
6  AFC Bournemouth 2022-10-29 14:00:00+00:00              3.000             1.200              0.200
7  AFC Bournemouth 2022-11-12 15:00:00+00:00              2.000             1.400              0.600


## 3 — GK composite score per team per match

In [4]:
# If multiple GKs appear (substitution), keep the one with more minutes
gk_match = (
    gk.sort_values('minutes_played', ascending=False)
    .drop_duplicates(subset=['match_id', 'team_name'], keep='first')
    [['match_id', 'team_name', 'composite_score', 'season']]
    .rename(columns={'composite_score': 'gk_composite'})
)

print(f'GK match records: {len(gk_match)} (unique match+team)')
missing = outfield[['match_id','team_name']].drop_duplicates().merge(
    gk_match[['match_id','team_name']], on=['match_id','team_name'], how='left', indicator=True)
print(f'Matches missing GK data: {(missing["_merge"] == "left_only").sum()}')


GK match records: 2032 (unique match+team)
Matches missing GK data: 71


## 4 — Outfield aggregation to team level

For each (match, team) compute:
- `mean_composite` — mean composite of all outfield players
- `weighted_composite` — minutes-weighted composite
- `att_composite` — mean composite of wingers + forwards
- `mid_composite` — mean composite of midfielders
- `def_composite` — mean composite of defenders
- `top3_composite` — mean of top 3 composites (key performers)
- `composite_std` — spread of composites (lineup cohesion)
- `n_defenders`, `n_midfielders`, `n_wingers`, `n_forwards` — formation shape


In [5]:
def team_agg(grp):
    cs  = grp['composite_score'].values
    mp  = grp['minutes_played'].values
    pos = grp['position_group'].values

    weighted = np.sum(cs * mp) / np.sum(mp) if np.sum(mp) > 0 else np.nan

    def pos_mean(p):
        mask = pos == p
        return cs[mask].mean() if mask.any() else np.nan

    top3 = np.sort(cs)[-3:].mean() if len(cs) >= 3 else cs.mean()

    return pd.Series({
        'mean_composite'  : cs.mean(),
        'weighted_composite': weighted,
        'att_composite'   : pos_mean('forward'),   # forward = clinical strikers
        'att2_composite'  : pos_mean('winger'),    # winger = wide attackers
        'mid_composite'   : pos_mean('midfielder'),
        'def_composite'   : pos_mean('defender'),
        'top3_composite'  : top3,
        'composite_std'   : cs.std() if len(cs) > 1 else 0.0,
        'n_defenders'     : int((pos == 'defender').sum()),
        'n_midfielders'   : int((pos == 'midfielder').sum()),
        'n_wingers'       : int((pos == 'winger').sum()),
        'n_forwards'      : int((pos == 'forward').sum()),
        'n_players'       : len(cs),
        # match context (same value for all players in team-match)
        'home_away'       : int(grp['home_away'].iloc[0]),
        'result'          : grp['result'].iloc[0],
        'season'          : grp['season'].iloc[0],
        'match_date'      : grp['match_date'].iloc[0],
        'home_team'       : grp['home_team'].iloc[0],
        'away_team'       : grp['away_team'].iloc[0],
    })


team_features = (
    outfield.groupby(['match_id', 'team_name'])
    .apply(team_agg)
    .reset_index()
)

print(f'Team-match features: {team_features.shape}')
print(team_features.head(4).to_string())


Team-match features: (2103, 21)
   match_id        team_name  mean_composite  weighted_composite  att_composite  att2_composite  mid_composite  def_composite  top3_composite  composite_std  n_defenders  n_midfielders  n_wingers  n_forwards  n_players  home_away result     season                match_date          home_team        away_team
0   3609937  Manchester City          -0.357              -0.357            NaN             NaN            NaN         -0.357          -0.357          0.000            1              0          0           0          1          0      D  2021_2022 2021-08-15 15:30:00+00:00  Tottenham Hotspur  Manchester City
1   3609938          Watford           0.157               0.157          0.157             NaN            NaN            NaN           0.157          0.000            0              0          0           1          1          1      W  2021_2022 2021-08-14 14:00:00+00:00            Watford      Aston Villa
2   3609939          Arsenal          

## 5 — Join GK composite and team strength

In [6]:
team_features = team_features.merge(
    gk_match[['match_id', 'team_name', 'gk_composite']],
    on=['match_id', 'team_name'], how='left'
)

team_features = team_features.merge(
    team_match[['match_id', 'team_name', 'roll5_attack_str', 'roll5_defence_str']],
    on=['match_id', 'team_name'], how='left'
)

# Fill missing GK composite with overall GK mean
gk_mean = gk['composite_score'].mean()
team_features['gk_composite'] = team_features['gk_composite'].fillna(gk_mean)

print('Nulls after join:')
print(team_features.isnull().sum()[team_features.isnull().sum() > 0])
print()
print(f'Total team-match rows: {len(team_features)}  ({team_features["match_id"].nunique()} matches)')


Nulls after join:
att_composite      75
att2_composite    177
mid_composite      25
def_composite      11
dtype: int64

Total team-match rows: 2103  (1059 matches)


## 6 — Pivot to one row per match
Split into home-team rows and away-team rows, then join on `match_id`.
Home features get prefix `h_`, away features get prefix `a_`.


In [7]:
FEAT_COLS = [
    'mean_composite', 'weighted_composite',
    'att_composite', 'att2_composite', 'mid_composite', 'def_composite',
    'top3_composite', 'composite_std', 'gk_composite',
    'n_defenders', 'n_midfielders', 'n_wingers', 'n_forwards',
    'roll5_attack_str', 'roll5_defence_str',
]

home_rows = team_features[team_features['home_away'] == 1].copy()
away_rows = team_features[team_features['home_away'] == 0].copy()

home_feat = home_rows[['match_id'] + FEAT_COLS + ['result', 'season', 'match_date',
                                                    'home_team', 'away_team']].copy()
away_feat = away_rows[['match_id'] + FEAT_COLS].copy()

home_feat = home_feat.rename(columns={c: f'h_{c}' for c in FEAT_COLS})
away_feat = away_feat.rename(columns={c: f'a_{c}' for c in FEAT_COLS})

match_df = home_feat.merge(away_feat, on='match_id', how='inner')

# Outcome from home team perspective: W→H, D→D, L→A
outcome_map = {'W': 'H', 'D': 'D', 'L': 'A'}
match_df['outcome'] = match_df['result'].map(outcome_map)
match_df = match_df.drop(columns=['result'])

print(f'Match-level dataset: {match_df.shape}')
print()
print('Outcome distribution:')
print(match_df['outcome'].value_counts(normalize=True).mul(100).round(1))
print()
print(match_df[['match_id','home_team','away_team','outcome',
                'h_mean_composite','a_mean_composite',
                'h_gk_composite','a_gk_composite']].head(6).to_string())


Match-level dataset: (1044, 36)

Outcome distribution:
outcome
A   35.200
H   34.700
D   30.200
Name: proportion, dtype: float64

   match_id               home_team         away_team outcome  h_mean_composite  a_mean_composite  h_gk_composite  a_gk_composite
0   3609939                 Arsenal           Chelsea       H             0.407             0.520           0.372          -0.009
1   3609940             Aston Villa  Newcastle United       A            -0.651             0.334           0.006           0.166
2   3609941  Brighton & Hove Albion           Watford       H             1.117             0.328           1.382          -2.022
3   3609945         Manchester City      Norwich City       H             1.076            -0.091           0.006           0.006
4   3609949             Aston Villa         Brentford       A             0.350             0.238           0.874          -1.078
5   3609950  Brighton & Hove Albion           Everton       H            -0.046           

## 7 — Sanity checks

In [8]:
# Outcome distribution per season
print('Outcome distribution per season:')
print(match_df.groupby('season')['outcome']
      .value_counts(normalize=True).mul(100).round(1)
      .unstack().to_string())
print()

# Feature nulls
null_check = match_df[[f'h_{c}' for c in FEAT_COLS] +
                       [f'a_{c}' for c in FEAT_COLS]].isnull().sum()
print('Feature nulls:')
print(null_check[null_check > 0] if null_check.any() else '  None — all clean ✓')
print()

# Mean composite differential — home vs away
match_df['composite_diff'] = match_df['h_mean_composite'] - match_df['a_mean_composite']
print('Composite differential by outcome (positive = home team stronger):')
print(match_df.groupby('outcome')['composite_diff'].mean().round(3))


Outcome distribution per season:
outcome        A      D      H
season                        
2021_2022 36.300 28.500 35.200
2022_2023 34.600 31.200 34.200
2023_2024 36.000 28.500 35.600
2024_2025 33.600 32.800 33.600

Feature nulls:
h_att_composite     35
h_att2_composite    86
h_mid_composite     12
h_def_composite      6
a_att_composite     36
a_att2_composite    86
a_mid_composite      9
a_def_composite      3
dtype: int64

Composite differential by outcome (positive = home team stronger):
outcome
A   -0.178
D   -0.024
H    0.172
Name: composite_diff, dtype: float64


In [9]:
from scipy.stats import pointbiserialr

# Encode outcome for correlation check
outcome_enc = match_df['outcome'].map({'H': 1, 'D': 0, 'A': -1})

print('Pearson r vs outcome encoding (H=1, D=0, A=-1):')
corr_rows = []
for col in [f'h_{c}' for c in FEAT_COLS] + [f'a_{c}' for c in FEAT_COLS]:
    valid = match_df[[col]].join(outcome_enc.rename('outcome_enc')).dropna()
    r = valid.corr().iloc[0, 1]
    corr_rows.append({'feature': col, 'r': r})

corr_df = pd.DataFrame(corr_rows).sort_values('r', key=abs, ascending=False)
print(corr_df.head(20).to_string(index=False))


Pearson r vs outcome encoding (H=1, D=0, A=-1):
             feature      r
    h_mean_composite  0.338
h_weighted_composite  0.319
    a_mean_composite -0.318
a_weighted_composite -0.299
    h_top3_composite  0.299
    a_top3_composite -0.286
     h_mid_composite  0.254
     a_mid_composite -0.252
  h_roll5_attack_str  0.242
    h_att2_composite  0.216
 h_roll5_defence_str -0.210
 a_roll5_defence_str  0.201
  a_roll5_attack_str -0.193
     h_composite_std  0.177
    a_att2_composite -0.177
      h_gk_composite -0.174
      a_gk_composite  0.172
     a_att_composite -0.166
     h_att_composite  0.156
     h_def_composite  0.145


## 8 — Train / test split and export

In [10]:
match_train = match_df[match_df['season'].isin(TRAIN_SEASONS)].reset_index(drop=True)
match_test  = match_df[match_df['season'].isin(TEST_SEASONS)].reset_index(drop=True)

print(f'Train matches: {len(match_train)}')
print(f'Test  matches: {len(match_test)}')
print()
print('Train outcome distribution:')
print(match_train['outcome'].value_counts())
print()
print('Test outcome distribution:')
print(match_test['outcome'].value_counts())

match_train.to_parquet(PROC_DIR / 'match_features_train.parquet', index=False)
match_test.to_parquet( PROC_DIR / 'match_features_test.parquet',  index=False)

for name, df in [('match_features_train', match_train), ('match_features_test', match_test)]:
    size = (PROC_DIR / f'{name}.parquet').stat().st_size / 1e3
    print(f'✓ {name}.parquet  ({len(df)} rows x {df.shape[1]} cols  {size:.0f} KB)')


Train matches: 794
Test  matches: 250

Train outcome distribution:
outcome
A    283
H    278
D    233
Name: count, dtype: int64

Test outcome distribution:
outcome
H    84
A    84
D    82
Name: count, dtype: int64
✓ match_features_train.parquet  (794 rows x 37 cols  176 KB)
✓ match_features_test.parquet  (250 rows x 37 cols  71 KB)
